# LLM News Research Tool
## Notebook 1: Finding and cleaning the news

This notebook covers the first stage of the tool: collecting news articles for a company. It uses two NewsAPI searches, combines their results, and removes duplicates, stock-listing pages and other unrelated pages. It also reads the full article wherever the website allows it. The result is a cleaned list of candidate articles for the next stages. Whether each article is relevant is checked later, in the upcoming stages.

Every table is computed from the committed data.

In [1]:
import json, os, subprocess, sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
from src.config import DATA_DIR, EVAL_DIR, RAW_DIR
from src.evaluate import is_relevant, load_label_set
from src.preprocess import normalize_url
from src.query import load_aliases

COMPANY = "Tata Motors"
LABEL_FILE = "v3_dev_tata_motors_checked.csv"
labels = load_label_set([EVAL_DIR / LABEL_FILE])
aliases = load_aliases(DATA_DIR / "company_aliases.json")
pd.set_option("display.max_colwidth", 90)

## 1. The data

The development data comes from eight NewsAPI searches for Tata Motors on 14, 23 and 24 October 2026. The same article can appear in more than one search, so the total number of records is higher than the number of unique articles.

Each unique article has a manual label based on labelling guide version 3. Articles labelled `direct` or `indirect` count as relevant. Relevant articles are also grouped by hand into events, so several reports about the same development count as one event.

Different sections use different article sets:

- **Choosing the searches** (section 2): articles from the three searches on 24 October.
- **Reading the full article** (section 4): all unique labelled articles.
- **Remaining errors** (section 6): articles kept on each search day. This means search A only on 14 and 23 October, and searches A and B on 24 October.

In [2]:
import csv
files, records = [], 0
for path in sorted(RAW_DIR.glob("*.json")):
    articles = json.loads(path.read_text(encoding="utf-8")).get("articles", [])
    labelled = [a for a in articles if normalize_url(a.get("url") or "") in labels]
    if labelled:
        files.append(path.name)
        records += len(labelled)

counts = Counter(labels.values())
events = {}
with open(EVAL_DIR / "events_dev_tata_motors.csv", encoding="utf-8-sig", newline="") as handle:
    for row in csv.DictReader(handle):
        events[normalize_url(row["url"])] = row["event"].strip().lower()
relevant_keys = [key for key, label in labels.items() if is_relevant(label)]

pd.DataFrame([
    ("Search files", len(files)),
    ("Article records in those searches", records),
    ("Unique labelled articles", len(labels)),
    ("  direct", counts["direct"]),
    ("  indirect", counts["indirect"]),
    ("  not", counts["not"]),
    ("Relevant articles (direct or indirect)", len(relevant_keys)),
    ("Relevant articles with an event", sum(key in events for key in relevant_keys)),
    ("Relevant events", len({events[key] for key in relevant_keys if key in events})),
], columns=["", "count"]).set_index("")

,count
,
Search files,8
Article records in those searches,206
Unique labelled articles,104
direct,26
indirect,21
not,57
Relevant articles (direct or indirect),47
Relevant articles with an event,47
Relevant events,24


## 2. Choosing the searches

Three searches are tested:

- **Search A:** looks for "Tata Motors".
- **Search B:** looks for subsidiary names listed in the alias file, including Jaguar Land Rover, JLR and the names of the two listed companies. It is a separate request with its own list of up to 30 results, so its articles do not take places in Search A's list.
- **Search C:** adds the company's short names to Search A.

The tool uses A + B. Search C and C + B are tested only during development. The comparison uses the articles returned by all three searches on 24 October.

Each search requests 30 articles from NewsAPI, which allows up to 100. This gives enough articles for the next steps without sending too many articles for language-model checks. The free plan also has a daily token limit.

In [3]:
from src.search_report import configuration_scores

record = json.loads((EVAL_DIR / "fresh_record_20261024.json").read_text(encoding="utf-8"))
responses = {name: json.loads((RAW_DIR / file).read_text(encoding="utf-8")) for name, file in record["raw_files"].items()}
search = configuration_scores(responses, labels)
table = pd.DataFrame.from_dict({name: {"requests": v["requests"], "articles returned": v["returned"],
                                       "relevant found": v["relevant"],
                                       "share of relevant in the pool": round(v["pool_recall"], 3),
                                       "precision": round(v["precision"], 3), "listing pages": v["listings"]}
                                for name, v in search["configurations"].items()}, orient="index")
print(f"Pool of 24 October: {search['pool_relevant']} relevant articles across the three searches.")
table

Pool of 24 October: 16 relevant articles across the three searches.


,requests,articles returned,relevant found,share of relevant in the pool,precision,listing pages
A,1,21,12,0.75,0.571,0
A+B,2,51,16,1.00,0.314,22
C,1,24,12,0.75,0.500,0
C+B,2,54,16,1.00,0.296,22


Search B needs one extra NewsAPI request, but it finds relevant articles that Search A misses. Search C does not find any additional relevant articles compared with Search A.

Search B also brings in more unrelated articles and stock-listing pages (see section 3). The four relevant articles found only by Search B are:

In [4]:
def relevant_in(name):
    found = {}
    for article in responses[name]["articles"]:
        url = normalize_url(article.get("url") or "")
        if url and is_relevant(labels.get(url, "")):
            found[url] = article
    return found

in_a, in_b = relevant_in("A"), relevant_in("B")
only_b = pd.DataFrame([{"title": a["title"], "source": (a.get("source") or {}).get("name", ""), "label": labels[url]}
                       for url, a in in_b.items() if url not in in_a])
with pd.option_context("display.max_colwidth", None):
    display(only_b)

,title,source,label
0,Better than the V8: Range Rover Electric tested,Autocar,direct
1,"Rawdon Glover Admits Jaguar Lost the Narrative With the Type 00, But Sees the Payoff in the Type 01",Observer,direct
2,Hard Drive: China forcing European car brands to cut back,RTE,indirect
3,"Jaguar Land Rover India expands retail presence in Delhi, Gurugram; sees NCR as key market",The Times of India,direct


Three of these articles cover Jaguar Land Rover. The fourth reports on China's pressure on European car brands to cut back.

## 3. Removing stock-listing pages

Search B also returns pages that only list share prices, such as "JLR share price today". Two checks remove these pages: a pattern in the title and a list of websites that publish them.

The website list catches all 22 listing pages. The title pattern catches 18 of those same pages. Neither check removes a relevant article.

In [5]:
from src.multi_search import merge
from src.search_report import listing_check_scores

listing = listing_check_scores(merge(responses, ["A", "B", "C"]), labels)

## 4. Reading the full article

NewsAPI provides a title, a short description and the first 200 characters of an article. Some articles mention the company only further down, so a filter using only this short text can miss them.

Stage 1 reads the full article when the website allows it. The tool uses the short text instead when the website blocks automatic reading, when the fetched page contains less text than the short NewsAPI text, or when the page does not match what NewsAPI says about the article (its title and description).

The full article text stays outside Git, in `data/fulltext/`. If this folder is missing, the notebook shows the saved result instead of running the comparison again.

In [6]:
from src.fulltext import article_id
from src.fulltext_eval import STAGE1_FILE, _inputs, compare

articles, labelled, full_texts = _inputs(LABEL_FILE)
FULL_TEXT_HERE = len(full_texts) > 0
if FULL_TEXT_HERE:
    result = compare(articles, labelled, full_texts, COMPANY, aliases)

    def events_kept(mode):
        kept = set(result[mode]["kept_ids"])
        all_events = {events[key] for key in relevant_keys if key in events}
        covered = {events[key] for key in relevant_keys if key in events and article_id(key) in kept}
        return f"{len(covered)}/{len(all_events)}"

    rows = {}
    for mode, name in (("short", "short text only"), ("full", "with full text")):
        r = result[mode]
        rows[name] = {"relevant kept": f"{r['tp']}/{r['tp'] + r['fn']}", "recall": round(r["recall"], 3),
                      "not relevant kept": r["fp"], "precision": round(r["precision"], 3),
                      "relevant events covered": events_kept(mode)}
    display(pd.DataFrame(rows).T)
else:
    saved = json.loads((EVAL_DIR / STAGE1_FILE).read_text(encoding="utf-8"))
    print("The full texts are not on this computer. The committed result (with full text):")
    display(pd.DataFrame(saved["records"]).set_index("label_file"))

,relevant kept,recall,not relevant kept,precision,relevant events covered
short text only,33/47,0.702,5,0.868,16/24
with full text,45/47,0.957,18,0.714,24/24


With full text, the tool keeps 45 of the 47 relevant articles, compared with 33 using short text alone. It also covers all 24 relevant events, compared with 16. However, it keeps 13 more articles that are not relevant, and precision falls from 0.868 to 0.714. Section 6 looks at where these extra articles end up.

## 5. Checking the cleaning

Each cleaning rule has a check that runs on the committed data. The notebook stops if any check fails.

In [7]:
from src.fulltext import FullTextFetcher
from src.fulltext_eval import usable_full_texts
from src.ranking_eval import DEV_RUNS, _load_run, stage1_candidates

status = Counter()
fetcher = FullTextFetcher()
for article in articles:
    path = fetcher._cache_path(article["url"])
    status[json.loads(path.read_text(encoding="utf-8")).get("status") if path.exists() else "not on this computer"] += 1

_, rejected = usable_full_texts(articles, full_texts, COMPANY, aliases)
duplicates = []
for name, run_files in DEV_RUNS.items():
    candidates = stage1_candidates(_load_run(run_files), full_texts, COMPANY, aliases)
    urls = [normalize_url(c["url"]) for c in candidates]
    duplicates.append((name, len(candidates), len(urls) - len(set(urls))))

ignored = subprocess.run(["git", "check-ignore", "data/fulltext/"], capture_output=True, text=True).returncode == 0
print("Pages of the labelled articles, by fetch status:", dict(status))
print("Fetched pages rejected by the page check:", len(rejected))
print("The same article twice among a day's candidates:", {name: dup for name, _, dup in duplicates})
print("Listing checks: relevant articles rejected =", listing["either"]["relevant_rejected"])
print("Publisher text kept out of Git (data/fulltext/ is ignored):", ignored)

Pages of the labelled articles, by fetch status: {'ok': 71, 'blocked_by_robots': 33}
Fetched pages rejected by the page check: 1
The same article twice among a day's candidates: {'2026-10-14': 0, '2026-10-23': 0, '2026-10-24': 0}
Listing checks: relevant articles rejected = 0
Publisher text kept out of Git (data/fulltext/ is ignored): True


In [8]:
# Each safeguard as an assertion on the real data: the notebook stops if one fails.
assert listing["either"]["relevant_rejected"] == 0, "a listing check rejects a relevant article"
assert all(dup == 0 for _, _, dup in duplicates), "the same article appears twice among a day's candidates"
assert ignored, "publisher text is not kept out of Git"
blocked = {a["url"] for a in articles if fetcher._cache_path(a["url"]).exists()
           and json.loads(fetcher._cache_path(a["url"]).read_text(encoding="utf-8")).get("status") != "ok"}
assert not blocked & set(full_texts), "a blocked page has full text"
usable, _ = usable_full_texts(articles, full_texts, COMPANY, aliases)
assert not set(rejected) & set(usable), "a page rejected by the sanity check is used"
if FULL_TEXT_HERE:
    saved = {r["label_file"]: r for r in json.loads((EVAL_DIR / STAGE1_FILE).read_text(encoding="utf-8"))["records"]}[LABEL_FILE]
    assert (result["full"]["tp"], result["full"]["fp"], result["full"]["fn"]) ==         (saved["relevant_kept"], saved["kept_not_relevant"], saved["relevant_missed"]), "the result differs from the committed record"
print("Every safeguard holds, and the full-text result equals the committed Stage 1 record.")

Every safeguard holds, and the full-text result equals the committed Stage 1 record.


Copies of one article are removed when they share the same cleaned web address or have nearly identical titles. Separate articles about the same development are not copies: they stay, and the event grouping counts them as one event.

## 6. Remaining errors

Stage 1 still misses two relevant articles. Both are labelled `indirect`, so no direct news about the company is lost.

In [9]:
if FULL_TEXT_HERE:
    notes = {}
    with open(EVAL_DIR / LABEL_FILE, encoding="utf-8-sig", newline="") as handle:
        for row in csv.DictReader(handle):
            notes[row["title"]] = (row["label"], row["note"])
    display(pd.DataFrame([{"missed relevant article": title[:90], "label": notes.get(title, ("", ""))[0],
                           "note": notes.get(title, ("", ""))[1]} for title in result["full"]["missed"]]))

,missed relevant article,label,note
0,Used Truck Market to Reach USD 70.74 Billion by 2031 as Digital Auctions and Fleet Ren...,indirect,"Global used-truck market report, Tata Motors only appears in a list of companies."
1,"Stellantis India aims to halve vehicle development time, boost localisation",indirect,


Stage 1 orders the kept articles by where they mention the company: the title first, then the description, then the article text. The table shows where the kept articles that are not relevant appear in this order.

In [10]:
from src import ranking as rk

rows = []
for name, run_files in DEV_RUNS.items():
    ordered = rk.order(stage1_candidates(_load_run(run_files), full_texts, COMPANY, aliases), "O1")
    places = [n for n, c in enumerate(ordered, start=1) if not is_relevant(labels[normalize_url(c["url"])])]
    rows.append({"search day": name, "candidates": len(ordered), "not relevant": len(places),
                 "in the top 5": sum(p <= 5 for p in places), "in the top 10": sum(p <= 10 for p in places),
                 "their places": places})
pd.DataFrame(rows).set_index("search day")

,candidates,not relevant,in the top 5,in the top 10,their places
search day,,,,,
2026-10-14,25,4,0,0,"[17, 19, 22, 25]"
2026-10-23,17,4,0,2,"[6, 10, 13, 15]"
2026-10-24,25,10,1,3,"[4, 6, 10, 11, 15, 17, 18, 20, 22, 25]"


Only one of them reaches the first 5 places, the part of the list read first.

## 7. Limitations

- NewsAPI's free Developer plan shows articles about one day late, searches at most one month back and allows 100 requests a day ([NewsAPI pricing](https://newsapi.org/pricing)).
- Some websites block automatic reading: 33 of the 104 labelled pages. Stage 1 respects this, so those articles are judged on their short text only.